# Memoria e Gestione dello Storico nelle Chat con LLM

# 1. Il Problema della Memoria

Un LLM è **stateless**: ogni chiamata API è indipendente, il modello non ricorda nulla di ciò che è stato detto in precedenza.

Per costruire una chat coerente occorre **ricostruire esplicitamente il contesto** a ogni turno, inviando la storia della conversazione insieme alla nuova domanda.

```
Turno 1:  [user: "Mi chiamo Marco"]         → LLM risponde
Turno 2:  [user: "Come mi chiamo?"]         → LLM NON sa (nessun contesto!)
Turno 2✓: [user: "Mi chiamo Marco"          → LLM SA (contesto incluso)
           assistant: "..."
           user: "Come mi chiamo?"]
```

## 1.1 Tipi di Memoria

| Tipo | Descrizione | Esempio |
|------|-------------|---------|
| **In-context** | Storia inclusa nel prompt | Lista messaggi |
| **A finestra** | Solo gli ultimi N scambi | Sliding window |
| **Compressa** | Storia riassunta dall'LLM | Summary memory |
| **Strutturata** | Fatti chiave estratti | Entity memory |
| **Semantica** | Recupero vettoriale (RAG) | ChromaDB |

## 1.2 Il Context Window

Ogni LLM ha un limite di token che può elaborare in una singola chiamata (es. 4096, 8192, 128k token).

$$\text{token usabili per risposta} = \text{context window} - \text{token della storia}$$

Discussione lunga → meno spazio per la risposta → l'LLM inizia a "dimenticare" i turni più vecchi.


# 2. Prerequisiti

```bash
pip install ollama chromadb
ollama pull ministral-3:3b
ollama pull all-minilm
```

In [1]:
import ollama
import chromadb
import json

MODEL = "ministral-3:3b"   # modello per la chat
EMBED = "all-minilm:latest"  # modello per gli embedding (sezione 7)

def generate_embedding(testo: str) -> list:
    response = ollama.embed(model=EMBED, input=testo)
    return response.embeddings[0]

print("Librerie caricate.")

Librerie caricate.


# 3. Livello 1 — Nessuna Memoria

Baseline: ogni chiamata è indipendente. Il modello risponde solo alla domanda corrente, senza alcun contesto precedente.

In [2]:
def chat_senza_memoria(domanda: str) -> str:
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": domanda}]
    )
    return response.message.content


# Demo: il modello non ricorda nulla tra le chiamate
r1 = chat_senza_memoria("Mi chiamo Marco e faccio il data scientist. Come stai?")
print("Turno 1:", r1[:120], "...")

r2 = chat_senza_memoria("Come mi chiamo e che lavoro faccio?")
print("\nTurno 2:", r2[:120], "...")
print("\n→ Il modello non sa chi è Marco: ogni chiamata è isolata.")

Turno 1: Grazie per avermi chiesto, Marco! Sono molto contento di conoscerti e di poter aiutarti come data scientist.

La mia ene ...

Turno 2: Sono un modello linguistico di grandi dimensioni creato da Mistral AI, un laboratorio di intelligenza artificiale con se ...

→ Il modello non sa chi è Marco: ogni chiamata è isolata.


# 4. Livello 2 — Storia Completa

## 4.1 Struttura dei Messaggi Ollama

`ollama.chat()` accetta una lista di messaggi con tre ruoli:

```python
[
    {"role": "system",    "content": "Sei un assistente utile."},  # opzionale
    {"role": "user",      "content": "Prima domanda"},
    {"role": "assistant", "content": "Prima risposta"},
    {"role": "user",      "content": "Seconda domanda"},
    ...
]
```

Ogni chiamata include l'intera storia: l'LLM "ricorda" simulando una conversazione continua.

## 4.2 Utility: visualizzare il contesto

In [3]:
def mostra_contesto(messaggi: list):
    """Stampa un riepilogo visivo dei messaggi nel contesto."""
    totale = sum(len(m['content']) for m in messaggi)
    print(f"{'Role':<12} {'Chars':>6}  Preview")
    print("─" * 65)
    for m in messaggi:
        anteprima = m['content'][:45].replace('\n', ' ')
        print(f"{m['role']:<12} {len(m['content']):>6}  {anteprima}…")
    print("─" * 65)
    print(f"{'TOTALE':<12} {totale:>6}  (~{totale // 4} token stimati)")

## 4.3 Chat con Storia Illimitata

In [4]:
def chat_con_storia(domanda: str, storia: list) -> tuple[str, list]:
    """Aggiunge ogni scambio alla storia. Nessun limite di dimensione."""
    storia.append({"role": "user", "content": domanda})

    response = ollama.chat(model=MODEL, messages=storia)
    risposta = response.message.content

    storia.append({"role": "assistant", "content": risposta})

    n_scambi   = len(storia) // 2
    n_chars    = sum(len(m['content']) for m in storia)
    print(f"[scambi: {n_scambi} | caratteri in contesto: {n_chars} | ~{n_chars//4} token]")
    return risposta, storia


storia = []
r1, storia = chat_con_storia("Mi chiamo Marco e lavoro come data scientist a Milano.", storia)
print("LLM:", r1[:120], "...")

r2, storia = chat_con_storia("Qual è il mio nome e dove lavoro?", storia)
print("\nLLM:", r2[:120], "...")

print("\n─ Contesto attuale ─")
mostra_contesto(storia)

[scambi: 1 | caratteri in contesto: 787 | ~196 token]
LLM: Ciao Marco! Sono lieto di conoscerti e di aiutarti con le tue esigenze legate al mondo della data science a Milano.

Che ...
[scambi: 2 | caratteri in contesto: 1008 | ~252 token]

LLM: **Nome:** Marco
**Lavoro:** Data Scientist a Milano

Se hai bisogno di approfondimenti su argomenti specifici (come stru ...

─ Contesto attuale ─
Role          Chars  Preview
─────────────────────────────────────────────────────────────────
user             54  Mi chiamo Marco e lavoro come data scientist …
assistant       733  Ciao Marco! Sono lieto di conoscerti e di aiu…
user             33  Qual è il mio nome e dove lavoro?…
assistant       188  **Nome:** Marco **Lavoro:** Data Scientist a …
─────────────────────────────────────────────────────────────────
TOTALE         1008  (~252 token stimati)


### Il Problema

La storia cresce senza limiti. Dopo molti scambi:
- Si supera il context window del modello
- I costi API aumentano (token inviati)
- Le risposte rallentano

**Soluzione**: limitare la quantità di storia inviata.

# 5. Livello 3 — Sliding Window (Finestra Scorrevole)

## 5.1 Concetto

Inviare solo gli **ultimi N scambi** anziché tutta la storia.

```
Storia completa:  [t1][t2][t3][t4][t5][t6][t7][t8]
Finestra (N=3):                       [t6][t7][t8]
```

**Pro**: contesto sempre sotto controllo  
**Contro**: gli scambi fuori finestra vengono dimenticati

## 5.2 Parametri

- `max_scambi`: numero di coppie user/assistant da mantenere
- `system_msg`: messaggio di sistema fisso (non consume finestra)

In [5]:
def sliding_window(storia: list, max_scambi: int = 5) -> list:
    """Mantieni gli ultimi max_scambi scambi completi + il messaggio user corrente.

    Il tag è necessario perché chat_finestra appende il messaggio user PRIMA di
    chiamare questa funzione: la lista ha sempre lunghezza dispari (2k + 1).
    Tagliare storia[-2k:] partirebbe da un messaggio assistant — finestra storta.
    Separare il pendente garantisce che la finestra inizi sempre con un user message.
    """
    pendente = storia[-1:]   # user message corrente (non ancora risposto)
    completa = storia[:-1]   # scambi già completati: sempre numero pari di messaggi

    max_msg  = max_scambi * 2
    finestra = completa[-max_msg:] if len(completa) > max_msg else completa
    return finestra + pendente


def chat_finestra(domanda: str, storia: list, max_scambi: int = 3,
                  system_msg: str = "") -> tuple[str, list]:
    """Chat con sliding window: il contesto inviato è sempre ≤ max_scambi scambi precedenti."""
    storia.append({"role": "user", "content": domanda})
    finestra = sliding_window(storia, max_scambi)

    messaggi = []
    if system_msg:
        messaggi.append({"role": "system", "content": system_msg})
    messaggi.extend(finestra)

    # Mostra cosa vede il modello PRIMA della chiamata
    n_tot   = len(storia) // 2
    passati = (len(finestra) - 1) // 2
    fuori   = n_tot - 1 - passati

    blocchi = []
    for i in range(1, n_tot):
        blocchi.append(f"░t{i}░" if i <= fuori else f"t{i}")
    blocchi.append(f"→t{n_tot}")
    print(f"  vede: {' '.join(blocchi)}")

    response = ollama.chat(model=MODEL, messages=messaggi)
    risposta = response.message.content
    storia.append({"role": "assistant", "content": risposta})

    return risposta, storia


# Demo
storia = []
win    = 3
print("Legenda:  tN = in contesto   ░tN░ = dimenticato   →tN = corrente\n")

scambi = [
    "Mi chiamo Marco, faccio il data scientist.",    # t1
    "Abito a Milano vicino al Duomo.",               # t2
    "Ho un gatto di nome Pixel.",                    # t3
    "Ho una sorella di nome Giulia.",                # t4
    "Come mi chiamo?",                               # t5 — t1 esce
    "Dimmi qualcosa di interessante sull'astronomia.", # t6 — filler
    "Come si chiama il mio gatto?",                  # t7 — t3 esce
]

for domanda in scambi:
    print(f"User: {domanda}")
    r, storia = chat_finestra(domanda, storia, win)
    print(f"LLM : {r[:120].replace(chr(10), ' ')}...")
    print()

Legenda:  tN = in contesto   ░tN░ = dimenticato   →tN = corrente

User: Mi chiamo Marco, faccio il data scientist.
  vede: →t0
LLM : Ciao Marco! Sono felice di conoscerti e di aiutarti come data scientist. 😊  Che tipo di aiuto o supporto hai bisogno ogg...

User: Abito a Milano vicino al Duomo.
  vede: →t1
LLM : Perfetto, Marco! Poiché abiti a Milano vicino al **Duomo**, posso aiutarti con alcune domande specifiche legate alla tua...

User: Ho un gatto di nome Pixel.
  vede: t1 →t2
LLM : Che bello che hai un gatto di nome **Pixel**! 🐾💻  Se vuoi, posso aiutarti anche in questo ambito, ad esempio: - **Analis...

User: Ho una sorella di nome Giulia.
  vede: t1 t2 →t3
LLM : Che bello avere una sorella come **Giulia**! 😊 Se vuoi, posso aiutarti con domande che coinvolgono sia te che Giulia, ad...

User: Come mi chiamo?
  vede: t1 t2 t3 →t4
LLM : Mi dispiace, ma non ho accesso ai tuoi dati personali o alla tua identità. Tuttavia, posso aiutarti con domande generali...

User: Dimmi qualcosa d

# 6. Livello 4 — Memoria con Riassunto

## 6.1 Approccio

Quando la storia supera una soglia, si usa l'**LLM stesso** per comprimerla in un riassunto. Il riassunto sostituisce i vecchi messaggi nel contesto.

```
Storia lunga:   [t1..t8]  →  riassunto: "L'utente si chiama Marco, ..."
                                              ↓
Contesto:  [system: riassunto] + [t7][t8] + [nuova domanda]
```

**Pro**: preserva le informazioni importanti, riduce i token  
**Contro**: può perdere dettagli, aggiunge una chiamata LLM extra

## 6.2 Funzione di Riassunto

In [6]:
def riassumi_storia(storia: list) -> str:
    """Usa l'LLM per condensare la storia in punti chiave."""
    testo = "\n".join([
        f"{'Utente' if m['role'] == 'user' else 'Assistente'}: {m['content']}"
        for m in storia
    ])
    response = ollama.chat(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": (
                "Riassumi questa conversazione in 4-6 punti chiave. "
                "Conserva nomi, fatti specifici e preferenze dell'utente.\n\n"
                f"{testo}"
            )
        }]
    )
    return response.message.content

## 6.3 Chat con Compressione Progressiva

In [7]:
def chat_riassunto(domanda: str, storia: list, riassunto: str = "",
                   max_scambi: int = 4, soglia: int = 6) -> tuple[str, list, str]:
    """
    Chat con compressione progressiva:
    - mantiene gli ultimi max_scambi scambi nel contesto live
    - comprime i turni più vecchi in un riassunto quando si supera soglia scambi
    - accumula i riassunti successivi
    """
    storia.append({"role": "user", "content": domanda})

    # Comprimi se la storia supera la soglia.
    # Al trigger, storia ha lunghezza dispari (user appena aggiunto).
    # Teniamo (max_scambi * 2 + 1) elementi per garantire che la fetta live
    # inizi sempre con un user message e includa il messaggio corrente.
    if len(storia) > soglia * 2:
        n_keep   = max_scambi * 2 + 1
        vecchia  = storia[:-n_keep]
        nuovo_r  = riassumi_storia(vecchia)
        riassunto = (f"{riassunto}\n\n[Aggiornamento riassunto]:\n{nuovo_r}".strip()
                     if riassunto else nuovo_r)
        storia   = storia[-n_keep:]
        print(f"  ↳ Compressione: {len(vecchia)//2} scambi → riassunto")

    # Costruisci messaggi
    messaggi = []
    if riassunto:
        messaggi.append({
            "role": "system",
            "content": f"Contesto della conversazione precedente:\n{riassunto}"
        })
    messaggi.extend(storia)

    response = ollama.chat(model=MODEL, messages=messaggi)
    risposta = response.message.content
    storia.append({"role": "assistant", "content": risposta})

    stato = "con riassunto" if riassunto else "senza riassunto"
    print(f"[scambi live: {len(storia)//2} | {stato}]")
    return risposta, storia, riassunto


storia, riassunto = [], ""

# Costruiamo una storia abbastanza lunga da triggerare la compressione
conversazione = [
    "Mi chiamo Marco, ho 32 anni e faccio il data scientist a Milano.",
    "Il mio linguaggio preferito è Python, uso molto pandas e sklearn.",
    "Ho un gatto di nome Pixel, è un europeo tigrato.",
    "Mi piace il calcio: sono tifoso dell'Inter.",
    "Sto studiando RAG e vector database per un progetto al lavoro.",
    "Ho una sorella che si chiama Giulia, studia medicina.",
    "Il mio film preferito è Inception.",
    "Qual è il mio nome, quanti anni ho e dove lavoro?",
]

for msg in conversazione:
    r, storia, riassunto = chat_riassunto(msg, storia, riassunto, max_scambi=4, soglia=5)

print("\nUltima risposta:", r[:200], "...")
if riassunto:
    print("\n─ Riassunto generato ─")
    print(riassunto[:400], "...")

[scambi live: 1 | senza riassunto]
[scambi live: 2 | senza riassunto]
[scambi live: 3 | senza riassunto]
[scambi live: 4 | senza riassunto]
[scambi live: 5 | senza riassunto]
  ↳ Compressione: 1 scambi → riassunto
[scambi live: 5 | con riassunto]
  ↳ Compressione: 1 scambi → riassunto
[scambi live: 5 | con riassunto]
  ↳ Compressione: 1 scambi → riassunto
[scambi live: 5 | con riassunto]

Ultima risposta: Ecco i dettagli che hai condiviso nel tuo contesto precedente:

### **Informazioni su di te:**
- **Nome:** Marco (non specificato esplicitamente, ma menzionato come utente principale).
- **Età:** **32 ...

─ Riassunto generato ─
Ecco il riassunto dei punti chiave della conversazione con Marco:

1. **Identità e contesto professionale**:
   Marco è un **data scientist di 32 anni** che lavora a **Milano**, una città attiva nel campo della data science, machine learning e intelligenza artificiale.

2. **Competenze e interessi tecnici**:
   Marco ha espresso interesse per domande specifich

# 7. Livello 5 — Memoria delle Entità

## 7.1 Concetto

Invece di comprimere la *storia*, si estraggono e mantengono i **fatti chiave** su persone, luoghi e preferenze in un dizionario strutturato.

```python
{
    "nome":       "Marco",
    "età":        "32 anni",
    "lavoro":     "data scientist a Milano",
    "animale":    "gatto di nome Pixel",
    "linguaggio": "Python"
}
```

Questo **profilo** viene iniettato nel system message ad ogni turno.

**Pro**: memoria compatta e strutturata, nessuna perdita di fatti espliciti  
**Contro**: cattura solo ciò che l'LLM riesce ad estrarre; non preserva il tono della conversazione

## 7.2 Estrazione con Ollama

In [8]:
def estrai_fatti(testo: str, fatti_attuali: dict) -> dict:
    """Usa l'LLM per aggiornare il profilo utente con le informazioni nel testo."""
    fatti_str = "\n".join([f"- {k}: {v}" for k, v in fatti_attuali.items()])

    response = ollama.chat(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": (
                "Analizza il testo e aggiorna le informazioni sull'utente.\n"
                f"Fatti già noti:\n{fatti_str or '(nessuno)'}\n\n"
                f"Testo: \"{testo}\"\n\n"
                "Restituisci SOLO le informazioni nuove o aggiornate, una per riga, "
                "nel formato  CHIAVE: valore  (chiavi in minuscolo, senza spazi).\n"
                "Se non ci sono nuove informazioni scrivi: NESSUNO"
            )
        }]
    )

    aggiornati = dict(fatti_attuali)
    for line in response.message.content.strip().splitlines():
        if ":" in line and "NESSUNO" not in line.upper():
            k, _, v = line.partition(":")
            k, v = k.strip().lower().replace(" ", "_"), v.strip()
            if k and v:
                aggiornati[k] = v
    return aggiornati

## 7.3 Chat con Profilo Utente

In [9]:
def chat_entita(domanda: str, storia_recente: list,
                fatti: dict, max_scambi: int = 4) -> tuple[str, list, dict]:
    """Chat che mantiene e aggiorna un profilo strutturato dell'utente."""

    # Aggiorna il profilo con la nuova domanda dell'utente
    fatti = estrai_fatti(domanda, fatti)

    # Costruisci il system message con il profilo
    profilo = "\n".join([f"- {k}: {v}" for k, v in fatti.items()])
    messaggi = []
    if profilo:
        messaggi.append({
            "role": "system",
            "content": f"Informazioni note sull'utente:\n{profilo}"
        })

    messaggi.extend(storia_recente[-max_scambi * 2:])
    messaggi.append({"role": "user", "content": domanda})

    response = ollama.chat(model=MODEL, messages=messaggi)
    risposta = response.message.content

    storia_recente.append({"role": "user",      "content": domanda})
    storia_recente.append({"role": "assistant", "content": risposta})

    return risposta, storia_recente, fatti


storia, fatti = [], {}

r1, storia, fatti = chat_entita(
    "Mi chiamo Marco, ho 32 anni, sono data scientist.", storia, fatti)

r2, storia, fatti = chat_entita(
    "Ho un gatto tigrato che si chiama Pixel.", storia, fatti)

r3, storia, fatti = chat_entita(
    "Sono tifoso dell'Inter.", storia, fatti)

r4, storia, fatti = chat_entita(
    "Cosa sai di me finora?", storia, fatti)

print("LLM:", r4[:300], "...")
print("\n─ Profilo estratto ─")
for k, v in fatti.items():
    print(f"  {k}: {v}")

LLM: Ecco un riassunto delle informazioni che ho raccolto finora su di te, Marco:

### **Dati personali e professionali**
- **Nome**: Marco
- **Età**: 32 anni
- **Professione**: Data scientist
- **Interessi/affiliazioni**:
  - Tifoso dell'**Inter** (San Siro, nerazzurro, tifo gatto con Pixel!)
  - Possib ...

─ Profilo estratto ─


# 8. Livello 6 — Memoria Semantica con RAG

## 8.1 Architettura

Invece di comprimere o filtrare la storia, si **indicizza ogni scambio** in un database vettoriale. Ad ogni nuova domanda si recuperano gli scambi passati **semanticamente rilevanti**.

```
Ogni scambio salvato:
  "Utente: Mi chiamo Marco..."  →  embedding  →  ChromaDB

Nuova domanda: "Come mi chiamo?"
  → embedding della domanda
  → ChromaDB recupera gli scambi più simili
  → scambi rilevanti inclusi nel system message
  → LLM risponde con il contesto giusto
```

**Pro**: scala a conversazioni lunghissime, recupera contesto rilevante anche da giorni fa  
**Contro**: non cattura la *sequenza* degli eventi, richiede un vector DB

## 8.2 Setup ChromaDB per la Memoria

In [ ]:
def init_memoria(path: str = "./chat_memory_vdb") -> chromadb.Collection:
    """Crea o carica il database vettoriale per la memoria della chat."""
    client     = chromadb.PersistentClient(path=path)
    collection = client.get_or_create_collection(
        name="memoria_chat",
        metadata={"hnsw:space": "cosine"}
    )
    return collection


def salva_scambio(collection, turno_id: int, user_msg: str, assistant_msg: str):
    """Salva uno scambio user/assistant come singolo documento."""
    documento  = f"Utente: {user_msg}\nAssistente: {assistant_msg}"
    embedding  = generate_embedding(documento)
    collection.add(
        embeddings=[embedding],
        documents=[documento],
        metadatas=[{"turno": turno_id, "anteprima_user": user_msg[:80]}],
        ids=[f"turno_{turno_id}"]
    )


def recupera_contesto_rilevante(collection, query: str, n: int = 3) -> list[str]:
    """Recupera gli scambi passati più simili semanticamente alla query."""
    if collection.count() == 0:
        return []
    query_emb = generate_embedding(query)
    risultati = collection.query(
        query_embeddings=[query_emb],
        n_results=min(n, collection.count())
    )
    return risultati['documents'][0]


# Test rapido di salvataggio e recupero
mem = init_memoria()
# Pulisci per demo
try:
    chromadb.PersistentClient(path="./chat_memory_vdb").delete_collection("memoria_chat")
    mem = init_memoria()
except Exception:
    pass

salva_scambio(mem, 1, "Mi chiamo Marco, faccio il data scientist.", "Ciao Marco!")
salva_scambio(mem, 2, "Ho un gatto di nome Pixel.",                  "Che bel nome!")
salva_scambio(mem, 3, "Sto studiando i vector database.",            "Topic interessante!")

# Recupera scambi rilevanti per una nuova domanda
contesti = recupera_contesto_rilevante(mem, "Qual è il mio nome?", n=2)
print("Scambi recuperati per 'Qual è il mio nome?':")
for i, ctx in enumerate(contesti, 1):
    print(f"  [{i}] {ctx[:80]}...")

## 8.3 Chat con Memoria RAG

In [ ]:
def chat_rag_memory(domanda: str, storia_recente: list,
                   collection, turno: list) -> tuple[str, list]:
    """
    Chat con memoria semantica:
    1. Recupera scambi passati rilevanti dal vector DB
    2. Li inserisce nel system message come contesto
    3. Mantiene solo gli ultimi 3 scambi nel contesto live
    4. Salva il nuovo scambio nel vector DB
    """
    # 1. Recupero semantico
    contesti  = recupera_contesto_rilevante(collection, domanda, n=3)

    # 2. Costruisci messaggi
    messaggi  = []
    if contesti:
        ctx_str = "\n\n---\n\n".join(contesti)
        messaggi.append({
            "role": "system",
            "content": f"Scambi precedenti rilevanti per questa domanda:\n{ctx_str}"
        })
    messaggi.extend(storia_recente[-6:])       # ultimi 3 scambi live
    messaggi.append({"role": "user", "content": domanda})

    # 3. Chiamata LLM
    response = ollama.chat(model=MODEL, messages=messaggi)
    risposta = response.message.content

    # 4. Salva il nuovo scambio
    turno[0] += 1
    salva_scambio(collection, turno[0], domanda, risposta)
    storia_recente.append({"role": "user",      "content": domanda})
    storia_recente.append({"role": "assistant", "content": risposta})

    print(f"[turno {turno[0]} | contesti RAG: {len(contesti)} | nel DB: {collection.count()}]")
    return risposta, storia_recente


# Demo: conversazione lunga con memoria RAG
try:
    chromadb.PersistentClient(path="./chat_memory_vdb").delete_collection("memoria_chat")
except Exception:
    pass
mem     = init_memoria()
storia  = []
turno   = [0]

scambi = [
    "Mi chiamo Marco, ho 32 anni.",
    "Lavoro come data scientist a Milano.",
    "Il mio linguaggio preferito è Python.",
    "Ho un gatto di nome Pixel.",
    "Sto imparando RAG e ChromaDB.",
    "Sono tifoso dell'Inter.",
    "Ho una sorella di nome Giulia che studia medicina.",
    # Domande che richiedono memoria di turni lontani
    "Qual è il mio nome e quanti anni ho?",
    "Come si chiama il mio gatto?",
    "Cosa sto studiando al lavoro?",
]

for msg in scambi:
    r, storia = chat_rag_memory(msg, storia, mem, turno)

print("\nUltima risposta:", r[:250], "...")

# 9. Sistema Completo

## 9.1 Confronto tra Approcci

| Approccio | Token usati | Memoria remota | Complessità | Ideale per |
|-----------|------------|---------------|-------------|------------|
| Nessuna memoria | Minimi | ✗ | Bassa | Test, Q&A singole |
| Storia completa | Crescenti | ✗ | Bassa | Chat brevi |
| Sliding window | Controllati | ✗ | Bassa | Chat medie |
| Riassunto | Medi | Parziale | Media | Chat lunghe |
| Entity memory | Bassi | Solo fatti | Media | Assistenti personali |
| RAG memory | Medi | ✓ Semantica | Alta | Chat molto lunghe |
| **Ibrido** | **Ottimali** | **✓ Completa** | **Alta** | **Produzione** |

## 9.2 Classe `ChatbotConMemoria`

Combina le strategie più efficaci:
- **Sliding window** per il contesto live recente
- **Riassunto** per comprimere la storia intermedia
- **RAG** per recuperare scambi lontani rilevanti

In [ ]:
class ChatbotConMemoria:
    """
    Chatbot con memoria a tre livelli:
    - Finestra scorrevole per il contesto immediato
    - Riassunto per la storia intermedia
    - RAG (ChromaDB) per la memoria a lungo termine
    """

    def __init__(self,
                 model:        str = "ministral-3:3b",
                 max_scambi:   int = 5,
                 soglia_compressione: int = 10,
                 vdb_path:     str = "./chatbot_completo_vdb"):
        self.model      = model
        self.max_scambi = max_scambi
        self.soglia     = soglia_compressione
        self.storia     = []
        self.riassunto  = ""
        self.turno      = 0

        # ChromaDB per memoria a lungo termine
        self._client = chromadb.PersistentClient(path=vdb_path)
        try:
            self._client.delete_collection("memoria_lt")
        except Exception:
            pass
        self.mem = self._client.get_or_create_collection(
            name="memoria_lt",
            metadata={"hnsw:space": "cosine"}
        )

    # ─── Metodi interni ────────────────────────────────────────

    def _comprimi(self):
        """Comprime la storia vecchia in un riassunto."""
        vecchia       = self.storia[:-self.max_scambi * 2]
        nuovo         = riassumi_storia(vecchia)
        self.riassunto = (f"{self.riassunto}\n\n{nuovo}".strip()
                          if self.riassunto else nuovo)
        self.storia   = self.storia[-self.max_scambi * 2:]
        print(f"  ↳ Compressione: {len(vecchia)//2} scambi → riassunto")

    def _build_messages(self, domanda: str) -> list:
        """Costruisce il payload messaggi per l'LLM."""
        contesti  = recupera_contesto_rilevante(self.mem, domanda, n=2)
        parti_sys = []

        if self.riassunto:
            parti_sys.append(f"Riassunto conversazione precedente:\n{self.riassunto}")
        if contesti:
            parti_sys.append("Scambi rilevanti recuperati dalla memoria:\n"
                             + "\n---\n".join(contesti))

        messaggi = []
        if parti_sys:
            messaggi.append({"role": "system", "content": "\n\n".join(parti_sys)})
        messaggi.extend(self.storia[-self.max_scambi * 2:])
        messaggi.append({"role": "user", "content": domanda})
        return messaggi

    # ─── API pubblica ──────────────────────────────────────────

    def chat(self, domanda: str) -> str:
        """Invia un messaggio e ottieni una risposta, aggiornando tutta la memoria."""
        self.turno += 1

        if len(self.storia) > self.soglia * 2:
            self._comprimi()

        messaggi = self._build_messages(domanda)
        response = ollama.chat(model=self.model, messages=messaggi)
        risposta = response.message.content

        self.storia.append({"role": "user",      "content": domanda})
        self.storia.append({"role": "assistant", "content": risposta})
        salva_scambio(self.mem, self.turno, domanda, risposta)

        return risposta

    def stato(self):
        """Stampa lo stato corrente della memoria."""
        print(f"Turno corrente  : {self.turno}")
        print(f"Storia live     : {len(self.storia)//2} scambi")
        print(f"Riassunto       : {'sì' if self.riassunto else 'no'}")
        print(f"Scambi nel RAG  : {self.mem.count()}")

## 9.3 Demo del Sistema Completo

In [ ]:
bot = ChatbotConMemoria(model=MODEL, max_scambi=4, soglia_compressione=6)

conversazione = [
    "Ciao! Mi chiamo Luca, ho 28 anni e studio machine learning.",
    "Lavoro part-time in una startup che fa computer vision.",
    "Il mio framework preferito è PyTorch.",
    "Ho una fidanzata di nome Sara, è ingegnera.",
    "Sto preparando una tesi sui transformer.",
    "Nel tempo libero gioco a scacchi online.",
    "Ho anche un blog dove scrivo di AI.",
    "La mia città è Torino.",
    # Domande su informazioni lontane nella storia
    "Ricordi come mi chiamo e cosa studio?",
    "Di cosa parla la mia tesi?",
    "Cosa faccio nel tempo libero?",
]

for msg in conversazione:
    print(f"\nUtente: {msg}")
    r = bot.chat(msg)
    print(f"Bot: {r[:180]}...")

print("\n" + "═"*55)
print("STATO FINALE DELLA MEMORIA")
print("═"*55)
bot.stato()

## 9.4 Riepilogo delle Strategie

```
Conversazione
     │
     ├── [scambi recenti] ──────────────────────── Sliding window
     │         ↓ (quando supera soglia)
     ├── [storia compressa] ──────────────────────  Riassunto LLM
     │
     └── [tutti gli scambi] → embedding → ChromaDB  RAG semantico
                                               ↓
                                     recupero su domanda
```

**Scelta dello strumento**:

- Chat breve / demo → `chat_finestra()`
- Assistente personale con profilo → `chat_entita()`
- Chat lunga single-session → `chat_riassunto()`
- Memoria persistente multi-sessione → `chat_rag_memory()`
- Produzione → `ChatbotConMemoria` (tutti e tre)
